# Home Credit Default Risk: Modular Machine Learning Project Demo

Welcome to the restructured, modular project repository! This notebook demonstrates how to load, preprocess, train, and evaluate the Home Credit model using the custom python module package from the `src/` directory. 

Instead of one massive, unstructured script, this structure follows production-level software design patterns by isolating configurations, data aggregations, sklearn transformers, pipeline construction, and utility scripts.

### 1. Import Project Modules

We import configurations, utility functions, custom transformers, and pipeline builders directly from the local package.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from src.config import TARGET, RANDOM_STATE
from src.utils import reduce_mem_usage, generate_mock_data
from src.preprocessing import preprocess_data
from src.pipeline import create_model_pipeline

print("All modules imported successfully!")

### 2. Generate and Clean Mock Datasets (Dry-Run)

We can generate a synthetic mock version of all 7 relational tables in a temporary folder. This allows us to dry-run the pipeline in seconds without loading a massive dataset.

In [ ]:
mock_data_dir = 'mock_data'
generate_mock_data(output_dir=mock_data_dir, sample_size=500)

# Load the application train data
df_train = pd.read_csv(os.path.join(mock_data_dir, 'application_train.csv'))
df_test = pd.read_csv(os.path.join(mock_data_dir, 'application_test.csv'))

print(f"Raw Train shape: {df_train.shape}")
print(f"Raw Test shape: {df_test.shape}")

### 3. Preprocess and Aggregate Relational Data

We merge train and test inputs and execute `preprocess_data`. This calls the modular preprocessing functions behind the scenes: aggregating bureaus, installment schedules, POS CASH records, credit card logs, and previous applications.

In [ ]:
df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_full = reduce_mem_usage(df_full, verbose=True)

# Aggregate supplemental tables and build interaction features
df_full = preprocess_data(mock_data_dir, df_full)

# Split back into train and test sets
train_processed = df_full[df_full[TARGET].notnull()].copy()
test_processed = df_full[df_full[TARGET].isnull()].copy()

print(f"Processed Train shape: {train_processed.shape}")
print(f"Processed Test shape: {test_processed.shape}")

### 4. Build and Train the Pipeline (CPU mode for demo)

We set up our model training inputs, split features and labels, construct the sklearn model pipeline containing custom transformers, and fit the CatBoost classifier.

In [ ]:
from sklearn.model_selection import train_test_split

X = train_processed.drop(columns=[TARGET, 'SK_ID_CURR'])
y = train_processed[TARGET]

# Split into train/validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

categorical_features = X_train.select_dtypes(include=['object', 'category', 'str', 'string']).columns.tolist()

# Create CatBoost pipeline forced to CPU for quick notebook demo
pipeline = create_model_pipeline(categorical_features, custom_params={'task_type': 'CPU', 'iterations': 300})

print("Training the modular model pipeline...")
X_val_transformed = pipeline[:-1].fit_transform(X_val)
pipeline.fit(X_train, y_train, classifier__eval_set=(X_val_transformed, y_val))
print("Training completed!")

### 5. Evaluate Performance

We predict probabilities on our validation split and output the ROC AUC score.

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)
print(f"Validation ROC AUC Score: {auc:.4f}")

### 6. Clean Up Mock Folders

Now we can clean up the temporary directory to keep our repository neat.

In [ ]:
import shutil
if os.path.exists(mock_data_dir):
    shutil.rmtree(mock_data_dir)
    print("Cleaned up mock data directory!")